In [ ]:
# ============================================================
# 0. LIBRARY IMPORT
# ============================================================
# numpy -> ithu oru maths library. Matrix, vector calculation ellam
# romba fast-a pannum.
# Intha full RNN-a naama PyTorch / TensorFlow illama, verum numpy
# vechu kaiyaala (from scratch) ezhuthurom. Appo than ulla enna
# nadakkuthu nu clear-a puriyum.

import numpy as np

In [ ]:
# ============================================================
# 1. TRAINING DATA
# ============================================================
# Ithu than namma model padikka pora data (training data).
# Chinna chinna sentences mattum than vechirukkom, beginner-ku easy-a
# irukkanum nu.
#
# Goal enna na: oru sentence-la munnadi vara words-a paathu,
# "adutha word enna varum?" nu model guess (predict) pannanum.
# Example: "I love" -> adutha word "English" or "Python" varalam.

sentences = [
    "I love English",
    "I hate English",
    "I love Python",
    "I hate Python",
    "I like machine learning"
]

In [ ]:
# ============================================================
# 2. BUILD VOCABULARY
# ============================================================
# Computer-ku words puriyathu, numbers mattum than puriyum.
# Athanala ovvoru unique word-kum oru number (ID) kudukkurom.
# Intha word list-a than "vocabulary" nu solluvom.

# set() -> duplicate words-a automatic-a remove pannidum.
# "I" 5 sentence-la vanthaalum, set-la oru thadava than irukkum.
words = set()

for sentence in sentences:
    # split() -> sentence-a space vechu words-a pirikkum.
    # "I love English" -> ["I", "love", "English"]
    # update() -> antha words ellathayum set-la add pannum.
    words.update(sentence.split())

# sorted() -> words-a alphabetical order-la arrange pannum.
# Ippadi panna, ovvoru thadava run pannalum same ID than kidaikkum.
words = sorted(words)

# word_to_id -> word kuduthaa, athoda number tharum.
# enumerate() -> (0, 'English'), (1, 'I') ... ippadi index + word tharum.
# Example: word_to_id["love"] -> 6
word_to_id = {word: i for i, word in enumerate(words)}

# id_to_word -> reverse. Number kuduthaa, word tharum.
# Model output number-a than tharum, athai thirumba word-a maatha ithu use aagum.
# Example: id_to_word[6] -> "love"
id_to_word = {i: word for word, i in word_to_id.items()}

# Mothathula ethana unique words irukku nu count.
# Output layer-la ethana options (words) irukkanum nu ithu than decide pannum.
vocab_size = len(words)

print("Vocabulary:")
print(word_to_id)
print("Vocabulary size:", vocab_size)

In [ ]:
# ============================================================
# 3. CREATE TRAINING EXAMPLES
# ============================================================
# Ovvoru sentence-layum irunthu "input -> answer" pairs create pannurom.
#
# "I love English" nu oru sentence irunthaa:
#
#   ["I"]          -> "love"      (I kku apram love varuthu)
#   ["I", "love"]  -> "English"   (I love kku apram English varuthu)
#
# Ippadi ovvoru step-layum "adutha word" enna nu model-ku solli
# kudukkurom. Ithu than next-word prediction.

# Ella (input, answer) pairs-um intha list-la store aagum.
training_data = []

for sentence in sentences:

    # Sentence-a words-a pirikkurom.
    sentence_words = sentence.split()

    # i = 1 la irunthu start pannurom, yen na kammiyaa 1 word
    # input-a venum, apram than adutha word predict panna mudiyum.
    for i in range(1, len(sentence_words)):

        # sentence_words[:i] -> 0 la irunthu i-1 varai ulla words (input).
        # i=1 -> ["I"],  i=2 -> ["I", "love"]
        input_words = sentence_words[:i]

        # sentence_words[i] -> input kku apram vara word (answer / target).
        target_word = sentence_words[i]

        # Words-a numbers (IDs) aa maathurom, yen na model numbers
        # mattum than purinjukkum.
        input_ids = [word_to_id[w] for w in input_words]
        target_id = word_to_id[target_word]

        # (input IDs, answer ID) pair-a list-la serkkurom.
        training_data.append((input_ids, target_id))


# Namma create panna examples correct-a irukka nu check panna,
# IDs-a thirumba words-a maathi print pannurom.
print("\nTraining examples:")

for x, y in training_data:
    print(
        [id_to_word[i] for i in x],
        "->",
        id_to_word[y]
    )

In [ ]:
# ============================================================
# 4. RNN PARAMETERS (WEIGHTS)
# ============================================================
# Ithu than model-oda "brain". Intha numbers (weights) than training-la
# konjam konjama maari, model-a smart aakkum.
# Start-la ellam random chinna numbers. Training panna panna correct
# values-ku nagarum.

# embedding_size -> ovvoru word-aiyum ethana numbers vechu represent
# pannanum. Inga 8 numbers vechu oru word-oda "meaning"-a store pannurom.
embedding_size = 8

# hidden_size -> RNN-oda "memory" size. Munnadi paatha words pathi
# ethana numbers-la nyabagam vechukkanum nu. Inga 16.
hidden_size = 16

# learning_rate -> ovvoru step-la weights-a evvalavu maathanum nu.
# Romba perusa vechaa jump panni miss pannidum, romba chinnatha
# vechaa romba slow-a padikkum.
learning_rate = 0.05

# Random number generator. seed=42 kuduthaa, ovvoru run-layum
# same random numbers varum -> results repeat pannalaam.
rng = np.random.default_rng(42)

# E -> Word embedding matrix. Shape: (vocab_size, embedding_size) = (8, 8)
# Ovvoru row-um oru word-oda vector. E[6] -> "love" word-oda 8 numbers.
# normal(0, 0.1, ...) -> 0 kitta ulla chinna random numbers.
E = rng.normal(0, 0.1, (vocab_size, embedding_size))

# Wx -> Input word vector-a hidden state-ku maathura weights.
# "Ippo vara word" evvalavu mukkiyam nu itha vechu decide aagum.
Wx = rng.normal(0, 0.1, (hidden_size, embedding_size))

# Wh -> Munnadi irukka hidden state (memory)-a pudhu hidden state-ku
# kondu pora weights. Ithu than RNN-oda special - pazhaya words-a
# nyabagam vechukkura part.
Wh = rng.normal(0, 0.1, (hidden_size, hidden_size))

# bh -> Hidden layer bias. Oru extra adjust panra value. Start-la 0.
bh = np.zeros(hidden_size)

# Wy -> Hidden state (memory)-la irunthu ovvoru word-kum oru score
# kudukkura weights. Shape: (vocab_size, hidden_size)
Wy = rng.normal(0, 0.1, (vocab_size, hidden_size))

# by -> Output layer bias. Start-la 0.
by = np.zeros(vocab_size)

In [ ]:
# ============================================================
# 5. SOFTMAX FUNCTION
# ============================================================
# Model ovvoru word-kum oru raw score tharum, like [2.1, -0.5, 3.4 ...].
# Intha scores-a paathu onnum puriyathu. Softmax atha
# probability (0 to 1, total = 1) aa maathum.
# Example: [0.1, 0.7, 0.2] -> 2nd word varathukku 70% chance.

def softmax(x):

    # Ella values-layum max value-a kazhikkurom.
    # Yen? exp() perusaana numbers-ku overflow (infinity) aagidum.
    # Max-a kazhichaa answer maaraathu, aana numbers safe-a irukkum.
    x = x - np.max(x)

    # exp() -> ella values-aiyum positive aakkum, perusa irukkuratha
    # innum perusa aakkum.
    exp_x = np.exp(x)

    # Total-aala divide panna, ella values-um sernthu 1 aagum -> probability.
    return exp_x / np.sum(exp_x)

In [ ]:
# ============================================================
# 6. FORWARD PASS
# ============================================================
# Forward pass -> input words-a model kulla anuppi, "adutha word enna"
# nu probability-a vaangurathu.
#
# RNN words-a ovvonna, order-la padikkum. Ovvoru word padikkum pothum
# athoda "memory" (h) update aagum. Kadaisi memory-a vechu
# adutha word-a guess pannum.

def forward(input_ids):

    # h -> hidden state (memory). Start-la ethuvum padikkala,
    # athanala ellam 0.
    h = np.zeros(hidden_size)

    # cache -> ovvoru step-la enna values vanthuchu nu save pannurom.
    # Backpropagation (learning) pannum pothu ithu thevai.
    cache = []

    # Ovvoru word-aiyum order-la eduthu process pannurom.
    for word_id in input_ids:

        # Word ID vechu, antha word-oda embedding vector-a edukkurom.
        # Example: word_id = 1 ("I") -> E[1] -> 8 numbers.
        x = E[word_id]

        # RNN-oda main formula:
        #
        #   h_t = tanh(Wx*x_t + Wh*h_(t-1) + bh)
        #
        # Wx @ x   -> ippo vara word-la irunthu info
        # Wh @ h   -> munnadi padicha words-oda memory
        # bh       -> bias
        # Moonaiyum add panni, tanh() pottu -1 to 1 kulla kondu varrom.
        # (@ -> matrix multiplication)
        h_new = np.tanh(
            Wx @ x +
            Wh @ h +
            bh
        )

        # Intha step-oda values-a save pannurom:
        # (word ID, word vector, pazhaya memory, pudhu memory)
        cache.append(
            (word_id, x, h, h_new)
        )

        # Pudhu memory-a adutha step-ku pass pannurom.
        h = h_new

    # Ella words-um padichathukku apram, kadaisi memory (h) vechu
    # ovvoru vocabulary word-kum oru score kanakku pannurom.
    #
    #   scores = Wy * h + by
    #
    scores = Wy @ h + by

    # Scores-a probabilities aa maathurom.
    probabilities = softmax(scores)

    # probabilities -> ovvoru word-um adutha word aa vara chance
    # h             -> kadaisi memory (backprop-ku venum)
    # cache         -> ella step values (backprop-ku venum)
    return probabilities, h, cache

In [ ]:
# ============================================================
# 7. BACKPROPAGATION THROUGH TIME (BPTT) - Learning part
# ============================================================
# Ithu than model "padikkura" idam.
#
# Steps:
#   1. Forward pass panni, model enna guess pannuthu nu paakkurom.
#   2. Loss -> model evvalavu thappa guess pannuchu nu oru number.
#   3. Gradients -> ovvoru weight-aiyum entha pakkam, evvalavu maathina
#      loss kammi aagum nu kandupudikkurom.
#   4. Weights-a konjam update pannurom.
#
# RNN-la time steps irukkarathaala, error-a kadaisi word-la irunthu
# first word varaikkum pinnaadi (backward) kondu porom.
# Athanala than "Through Time" nu per.

def train_example(input_ids, target_id):

    # Function kulla weights-a maathurom, athu veliya ulla actual
    # variables-a maathanum. Athukku than "global".
    global E, Wx, Wh, bh, Wy, by

    # -------------------------
    # Forward pass
    # -------------------------
    # Model-oda current guess-a vaangurom.

    probabilities, final_h, cache = forward(input_ids)

    # Cross entropy loss:
    #   loss = -log(correct word-oda probability)
    # Correct word-ku probability 1 kitta irunthaa loss 0 kitta varum (nallathu).
    # Probability 0 kitta irunthaa loss romba perusa aagum (mosamaana guess).
    # 1e-12 -> log(0) = -infinity aagama irukka oru chinna safety number.
    loss = -np.log(probabilities[target_id] + 1e-12)


    # -------------------------
    # Gradients
    # -------------------------
    # Ovvoru weight-kum oru gradient (d...) variable create pannurom.
    # Same shape, start-la ellam 0. Apram ithula values serpom.
    # dE  -> E-oda gradient, dWx -> Wx-oda gradient ... ippadi.

    dE = np.zeros_like(E)

    dWx = np.zeros_like(Wx)
    dWh = np.zeros_like(Wh)

    dbh = np.zeros_like(bh)

    dWy = np.zeros_like(Wy)
    dby = np.zeros_like(by)

    # -------------------------
    # Output gradient
    # -------------------------
    # Softmax + cross entropy sernthaa gradient romba simple:
    #   d_scores = probabilities - correct_answer (one-hot)
    # Athaavathu, correct word position-la mattum 1 kazhikkurom.
    # Example: correct word-ku prob 0.3 na -> 0.3 - 1 = -0.7
    #          (negative -> "intha score-a innum athigam pannu")
    #          Matha words positive -> "avangaloda score-a kammi pannu"

    d_scores = probabilities.copy()

    d_scores[target_id] -= 1

    # Wy-oda gradient = error x kadaisi memory.
    # np.outer -> rendu vector-a vechu oru matrix create pannum.
    dWy += np.outer(d_scores, final_h)

    # Output bias-oda gradient = error thaan.
    dby += d_scores

    # Error-a output layer-la irunthu hidden state (memory)-ku
    # pinnaadi anuppurom. Wy.T -> Wy-oda transpose (reverse direction).
    dh = Wy.T @ d_scores


    # -------------------------
    # Backprop through time
    # -------------------------
    # reversed(cache) -> kadaisi word-la irunthu first word varaikkum
    # pinnaadi porom. Ovvoru step-layum, antha step weights-ku
    # evvalavu blame nu kanakku pannurom.

    for word_id, x, h_previous, h_current in reversed(cache):

        # tanh-oda derivative = 1 - tanh(x)^2
        # h_current already tanh output thaan, athanala direct-a use pannurom.
        # dz -> tanh-ku munnadi irukka value-oda gradient.
        dz = dh * (1 - h_current ** 2)

        # Hidden bias gradient. "+=" yen na ovvoru time step-layum
        # same bh than use aaguthu, so ellam serththu add pannurom.
        dbh += dz

        # Wx gradient = error x input word vector.
        dWx += np.outer(dz, x)

        # Wh gradient = error x munnadi irunthu memory.
        dWh += np.outer(dz, h_previous)

        # Antha word-oda embedding row-ku mattum gradient.
        # Matha words-oda embedding intha step-la use aagala, so maaraathu.
        dE[word_id] += Wx.T @ dz

        # Error-a innum oru step pinnaadi (munnadi word-ku) anuppurom.
        # Wh vazhiya than pazhaya memory pudhu memory-a affect pannuchu,
        # athanala Wh.T vechu thirumba anuppurom.
        dh = Wh.T @ dz


    # -------------------------
    # Gradient clipping
    # -------------------------
    # RNN-la sila neram gradients romba romba perusa aagidum
    # ("exploding gradients"). Appo weights pichikittu poidum.
    # Athai thadukka, ella gradient values-aiyum -5 to 5 kulla
    # adakkurom. out=gradient -> same array-laye maathum (new copy illa).

    for gradient in [dE, dWx, dWh, dbh, dWy, dby]:

        np.clip(
            gradient,
            -5,
            5,
            out=gradient
        )


    # -------------------------
    # Update weights
    # -------------------------
    # Gradient descent:
    #   pudhu weight = pazhaya weight - learning_rate * gradient
    # Gradient "loss athigam aagura direction" kaattum, athanala
    # minus pottu opposite direction-la porom -> loss kammi aagum.

    E -= learning_rate * dE

    Wx -= learning_rate * dWx
    Wh -= learning_rate * dWh
    bh -= learning_rate * dbh

    Wy -= learning_rate * dWy
    by -= learning_rate * dby

    # Intha example-oda loss-a return pannurom, progress track panna.
    return loss

In [ ]:
# ============================================================
# 8. TRAINING LOOP
# ============================================================
# Ella training examples-aiyum model-ku thirumba thirumba kaattrom.
# Oru thadava full data-vum paathaa, athu oru "epoch".
# 5000 epochs -> full data-va 5000 thadava padikkum.

epochs = 5000

for epoch in range(epochs):

    # Intha epoch-la ella examples-oda loss-aiyum serththu paakka.
    total_loss = 0

    # Shuffle -> examples order-a random-a maathurom.
    # Yen? Same order-la kaattinaa, model antha order-a manappaadam
    # pannidum. Mix panna nalla generalize aagum.
    rng.shuffle(training_data)

    for input_ids, target_id in training_data:

        # Ovvoru example-kum: forward + backward + weight update.
        loss = train_example(
            input_ids,
            target_id
        )

        total_loss += loss

    # Ovvoru 500 epochs-kum oru thadava loss print pannurom.
    # Loss kammi aagitte vanthaa -> model padikkuthu nu artham.
    # Note: "I love" -> English/Python rendume correct answer,
    # athanala loss full-a 0 aagathu. Athu normal thaan.
    if epoch % 500 == 0:

        print(
            f"Epoch {epoch:4d} "
            f"Loss = {total_loss:.4f}"
        )

In [ ]:
# ============================================================
# 9. PREDICT NEXT WORD
# ============================================================
# Training mudinjathukku apram, oru pudhu text kuduthu
# "adutha word enna?" nu model-a kekkurom.

def predict_next_word(text):

    # Text-a words-a pirikkurom. "I love" -> ["I", "love"]
    words = text.split()

    # Words-a IDs aa maathurom.
    # Note: vocabulary-la illatha word kuduthaa KeyError varum.
    input_ids = [
        word_to_id[w]
        for w in words
    ]

    # Forward pass mattum pothum (ippo padikkala, just guess pannurom).
    # _ , _ -> h, cache namakku ippo thevai illa, athanala ignore.
    probabilities, _, _ = forward(input_ids)

    # argmax -> athigamaana probability irukka word-oda ID.
    best_id = np.argmax(probabilities)

    print("\nInput:", text)

    print("\nPredictions:")

    # Ella words-aiyum probability athigam -> kammi order-la kaatta:
    # argsort() -> chinnathu la irunthu perusu varaikkum IDs tharum,
    # [::-1] -> athai reverse panni perusu first varum.
    sorted_ids = np.argsort(probabilities)[::-1]

    for word_id in sorted_ids:

        # :12s -> word-a 12 characters width-la align panni print.
        # :.4f -> probability-a 4 decimal places-la print.
        print(
            f"{id_to_word[word_id]:12s}"
            f" {probabilities[word_id]:.4f}"
        )

    print(
        "\nPredicted next word:",
        id_to_word[best_id]
    )

In [ ]:
# ============================================================
# 10. TEST
# ============================================================
# "I hate" kku apram training data-la "English" um "Python" um
# rendume vanthuchu. Athanala model rendukkum kittatta 50% - 50%
# kudukkum. Model correct-a thaan padichirukku nu artham.

predict_next_word("I hate")

In [ ]:
# "I love" kum same thaan - "English" / "Python" rendume possible,
# so rendukkum kittatta 50% varum.

predict_next_word("I love")

In [ ]:
# "I like" kku apram data-la eppovume "machine" thaan vanthuchu.
# Athanala model "machine" kku kittatta 100% confidence kudukkum.

predict_next_word("I like")